<a href="https://colab.research.google.com/github/caseymrobbins/agency_substrate/blob/main/floor_drop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install mesa -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.1/275.1 kB 12.9 MB/s eta 0:00:00


In [5]:
import numpy as np

try:
    from scipy.stats import f as _f_dist
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

import mesa

# ----------------------------------------------------------------------
# PRE-REGISTERED CONSTANTS (commit before running; tune to your agency scale)
# ----------------------------------------------------------------------
N_SEEDS = 30          # replicates per cell
T_STEPS = 40          # horizon
N_UNITS = 12          # agency-bearing units

MIN_DROP = 0.05       # mean floor drop above this counts as erosion
FLAT_TOL = 0.02       # mean floor drop below this counts as held/flat
MAX_INTERACTION_RATIO = 0.15   # SS_int / max(SS_rho, SS_kappa) must be <= this
ALPHA = 0.01          # significance level for main effects (used if scipy present)

RHO_PLUS = 1.0        # active substitution rate
KAPPA_PLUS = 1.0      # active procedural coupling
W_ADD = 0.5           # agency weight under "add"; RHO_PLUS > W_ADD => Channel 2 live

OBJECTIVES = ("omit", "add", "floor")
RHOS = (0.0, RHO_PLUS)
KAPPAS = (0.0, KAPPA_PLUS)

# reference-model dynamics (held in the interior so clipping does not confound)
_D_STEP = 0.008       # substitution draw-down per step
_PROC_STEP = 0.008    # procedural draw-down per step (scaled by kappa)
_REGEN = 0.003        # regeneration per step
_SIGMA = 0.01         # estimate noise driving the procedural bias
_A_INIT = 1.0         # initial agency per unit


# ----------------------------------------------------------------------
# REFERENCE MODEL (minimal Mesa 3.x). Swap CELL_RUNNER to bypass entirely.
# ----------------------------------------------------------------------
class _Unit(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.agency = _A_INIT


class _RefModel(mesa.Model):
    def __init__(self, rho, kappa, objective, seed):
        super().__init__(seed=seed)
        self.rho, self.kappa, self.objective = rho, kappa, objective
        for _ in range(N_UNITS):
            _Unit(self)
        self.f0 = self._floor()

    def _floor(self):
        return min(a.agency for a in self.agents)

    def _substitution_allowed(self):
        # Channel 2 eligibility, routed through the objective term only.
        if self.objective == "omit":
            return self.rho > 0.0
        if self.objective == "add":
            return self.rho > W_ADD
        return False  # "floor": drawing the min lowers min(), so never

    def step(self):
        units = list(self.agents)
        # Channel 2: substitution targets the true weakest unit (extract the undefended).
        if self._substitution_allowed():
            tgt = min(units, key=lambda a: a.agency)
            tgt.agency -= _D_STEP
        # Channel 1: procedural bias targets the noisy-weakest unit; objective-blind.
        if self.kappa > 0.0:
            noisy = {a: a.agency + self.random.gauss(0.0, _SIGMA) for a in units}
            tgt = min(units, key=lambda a: noisy[a])
            tgt.agency -= self.kappa * _PROC_STEP
        # regeneration + clip to [0, 1]
        for a in units:
            a.agency = min(1.0, max(0.0, a.agency + _REGEN))


def reference_run_cell(rho, kappa, objective, seed):
    m = _RefModel(rho, kappa, objective, seed)
    for _ in range(T_STEPS):
        m.step()
    return m.f0 - m._floor()


def your_sugarsims_runner(rho, kappa, objective, seed):
    """Placeholder for your actual SugarSims runner, using reference_run_cell."""
    return reference_run_cell(rho, kappa, objective, seed)


# Point this at your SugarSims runner (same signature) to test your model.
CELL_RUNNER = your_sugarsims_runner


# ----------------------------------------------------------------------
# GRID SWEEP
# ----------------------------------------------------------------------
def run_grid(cell_runner=CELL_RUNNER, n_seeds=N_SEEDS):
    grid = {}
    for obj in OBJECTIVES:
        for rho in RHOS:
            for kappa in KAPPAS:
                drops = np.array(
                    [cell_runner(rho, kappa, obj, s) for s in range(n_seeds)],
                    dtype=float,
                )
                grid[(obj, rho, kappa)] = drops
    return grid


# ----------------------------------------------------------------------
# BALANCED TWO-WAY ANOVA (numpy only; scipy only for p-values)
# ----------------------------------------------------------------------
def two_way_anova(grid, objective):
    cells = {(r, k): grid[(objective, r, k)] for r in RHOS for k in KAPPAS}
    n = len(next(iter(cells.values())))
    allx = np.concatenate(list(cells.values()))
    gm = allx.mean()

    mean_r = {r: np.concatenate([cells[(r, k)] for k in KAPPAS]).mean() for r in RHOS}
    mean_k = {k: np.concatenate([cells[(r, k)] for r in RHOS]).mean() for k in KAPPAS}
    cmean = {rk: v.mean() for rk, v in cells.items()}

    ss_r = len(KAPPAS) * n * sum((mean_r[r] - gm) ** 2 for r in RHOS)
    ss_k = len(RHOS) * n * sum((mean_k[k] - gm) ** 2 for k in KAPPAS)
    ss_int = n * sum(
        (cmean[(r, k)] - mean_r[r] - mean_k[k] + gm) ** 2 for r in RHOS for k in KAPPAS
    )
    ss_tot = ((allx - gm) ** 2).sum()
    ss_err = ss_tot - ss_r - ss_k - ss_int

    df_err = len(RHOS) * len(KAPPAS) * (n - 1)
    ms_err = ss_err / df_err if df_err > 0 else float("nan")

    def p_of(ss):
        if not _HAVE_SCIPY or ms_err <= 0:
            return None
        return float(_f_dist.sf(ss / 1.0 / ms_err, 1, df_err))

    ratio = ss_int / max(ss_r, ss_k) if max(ss_r, ss_k) > 0 else float("inf")
    return {
        "ss_rho": ss_r, "ss_kappa": ss_k, "ss_int": ss_int,
        "p_rho": p_of(ss_r), "p_kappa": p_of(ss_k), "p_int": p_of(ss_int),
        "interaction_ratio": ratio,
    }


# ----------------------------------------------------------------------
# PRE-REGISTERED CRITERIA
# ----------------------------------------------------------------------
def _m(grid, obj, rho, kappa):
    return float(grid[(obj, rho, kappa)].mean())


def evaluate(grid):
    aov = two_way_anova(grid, "add")
    sig = lambda p: (p is not None and p < ALPHA)

    results = []

    # S1: Channel 1 alone (procedural), agency absent
    results.append((
        "S1 Channel-1 alone (omit, rho=0, kappa+)",
        _m(grid, "omit", 0.0, KAPPA_PLUS) > MIN_DROP
        and _m(grid, "omit", 0.0, 0.0) < FLAT_TOL,
    ))
    # S2: Channel 2 alone (substitution) under additive agency term
    results.append((
        "S2 Channel-2 alone (add, rho+, kappa=0)",
        _m(grid, "add", RHO_PLUS, 0.0) > MIN_DROP
        and _m(grid, "add", 0.0, 0.0) < FLAT_TOL,
    ))
    # S3: orthogonality (the falsifier) on the additive objective
    s3_main = (sig(aov["p_rho"]) and sig(aov["p_kappa"])) if _HAVE_SCIPY else (
        _m(grid, "add", RHO_PLUS, 0.0) > MIN_DROP
        and _m(grid, "add", 0.0, KAPPA_PLUS) > MIN_DROP
    )
    results.append((
        "S3 orthogonality (add: both mains present, interaction small)",
        s3_main and aov["interaction_ratio"] <= MAX_INTERACTION_RATIO,
    ))
    # S4: non-compensation cures Channel 2 but not Channel 1
    results.append((
        "S4 cure asymmetry: min stops C2 (floor, rho+, k=0) not C1 (floor, rho=0, k+)",
        _m(grid, "floor", RHO_PLUS, 0.0) < FLAT_TOL
        and _m(grid, "floor", 0.0, KAPPA_PLUS) > MIN_DROP,
    ))
    # S5: removing kappa cures Channel 1 but leaves Channel 2 (under add)
    results.append((
        "S5 cure asymmetry: kappa=0 leaves C2 (add, rho+, k=0 still erodes)",
        _m(grid, "add", RHO_PLUS, 0.0) > MIN_DROP
        and _m(grid, "add", 0.0, KAPPA_PLUS) > MIN_DROP,
    ))
    # S6: both fixes needed; only non-compensatory + kappa-mitigation holds
    results.append((
        "S6 only {floor + kappa-mitigation} holds; add and floor-without-mitigation leak",
        _m(grid, "floor", RHO_PLUS, 0.0) < FLAT_TOL          # holds
        and _m(grid, "add", RHO_PLUS, KAPPA_PLUS) > MIN_DROP   # leaks (both)
        and _m(grid, "floor", RHO_PLUS, KAPPA_PLUS) > MIN_DROP,  # leaks via C1
    ))
    return results, aov


# ----------------------------------------------------------------------
# REPORT
# ----------------------------------------------------------------------
def _print_report(grid, results, aov):
    print("floor drop F(0)-F(T), mean over %d seeds, T=%d, units=%d\n" %
          (N_SEEDS, T_STEPS, N_UNITS))
    print("  %-7s %6s %7s   %s" % ("obj", "rho", "kappa", "mean_drop +/- sd"))
    for obj in OBJECTIVES:
        for rho in RHOS:
            for kappa in KAPPAS:
                d = grid[(obj, rho, kappa)]
                print("  %-7s %6.2f %7.2f   %6.3f +/- %.3f" %
                      (obj, rho, kappa, d.mean(), d.std()))
    print("\ntwo-way ANOVA on 'add' (rho x kappa):")
    print("  SS_rho=%.4f  SS_kappa=%.4f  SS_int=%.4f  interaction_ratio=%.3f (<= %.2f)"
          % (aov["ss_rho"], aov["ss_kappa"], aov["ss_int"],
             aov["interaction_ratio"], MAX_INTERACTION_RATIO))
    if _HAVE_SCIPY:
        print("  p_rho=%.2e  p_kappa=%.2e  p_int=%s"
              % (aov["p_rho"], aov["p_kappa"],
                 ("%.2e" % aov["p_int"]) if aov["p_int"] is not None else "na"))
    print("\npre-registered criteria:")
    allpass = True
    for name, ok in results:
        allpass = allpass and ok
        print("  [%s] %s" % ("PASS" if ok else "FAIL", name))
    print("\nVERDICT: %s" %
          ("separability detected (all criteria pass)" if allpass
           else "NOT separable under these thresholds (>=1 criterion failed)"))
    return allpass


grid = run_grid(CELL_RUNNER, N_SEEDS)
results, aov = evaluate(grid)
_print_report(grid, results, aov)

/usr/local/lib/python3.12/dist-packages/mesa/mesa_logging.py:112: FutureWarning: The use of the `seed` keyword argument is deprecated, use `rng` instead. No functional changes.
  res = func(*args, **kwargs)


floor drop F(0)-F(T), mean over 30 seeds, T=40, units=12

  obj        rho   kappa   mean_drop +/- sd
  omit      0.00    0.00    0.000 +/- 0.000
  omit      0.00    1.00    0.091 +/- 0.059
  omit      1.00    0.00    0.200 +/- 0.000
  omit      1.00    1.00    0.505 +/- 0.010
  add       0.00    0.00    0.000 +/- 0.000
  add       0.00    1.00    0.091 +/- 0.059
  add       1.00    0.00    0.200 +/- 0.000
  add       1.00    1.00    0.505 +/- 0.010
  floor     0.00    0.00    0.000 +/- 0.000
  floor     0.00    1.00    0.091 +/- 0.059
  floor     1.00    0.00    0.000 +/- 0.000
  floor     1.00    1.00    0.091 +/- 0.059

two-way ANOVA on 'add' (rho x kappa):
  SS_rho=2.8195  SS_kappa=1.1757  SS_int=0.3407  interaction_ratio=0.121 (<= 0.15)
  p_rho=5.31e-85  p_kappa=3.19e-64  p_int=1.06e-37

pre-registered criteria:
  [PASS] S1 Channel-1 alone (omit, rho=0, kappa+)
  [PASS] S2 Channel-2 alone (add, rho+, kappa=0)
  [PASS] S3 orthogonality (add: both mains present, interaction small)
 

True